# Vision Transformer: CNN feature map에서 patch token까지

Vision Transformer(ViT)는 이미지를 고정 크기 patch로 나누고, 각 patch를 token 벡터로 바꿔 Transformer encoder에 입력하는 비전 모델이다. CNN이 kernel을 주변 픽셀에 반복 적용해 지역 구조를 강하게 반영한다면, ViT는 self-attention으로 모든 token 사이의 관계를 각 encoder 층에서 계산한다.

이 노트북은 CNN의 `(B, C, H, W)` 표현과 ViT의 `(B, N+1, D)` 표현 사이의 경계를 먼저 확인한다. 이어서 patch projection, `[CLS]` token, positional embedding, self-attention의 역할을 작은 CPU 코드로 검증하고, 마지막에 실제 사전학습 모델의 전처리와 분류 경로를 연결한다.

## ViT 구조도

입력 이미지를 같은 크기의 patch로 자르고, 각 patch를 펼쳐 선형 projection한 뒤, 맨 앞에 학습 가능한 classification token을 붙인다.

이 sequence에 positional embedding을 더해 Transformer encoder로 보내고, 마지막 `[CLS]` 표현을 classification head에 전달한다.

<img src="https://raw.githubusercontent.com/google-research/vision_transformer/main/vit_figure.png" width="1100" alt="Vision Transformer의 patch projection, positional embedding, Transformer encoder와 classification head 구조"/>

출처: [Google Research vision_transformer 저장소](https://github.com/google-research/vision_transformer), 라이선스: [Apache License 2.0](https://github.com/google-research/vision_transformer/blob/main/LICENSE)이다.

### 1. 이미지를 token sequence로 바꾸는 부분

- **Patch**: 입력 이미지를 일정한 크기로 나눈 작은 이미지 조각
- **Flatten**: patch의 픽셀값을 한 줄의 벡터로 펼치는 연산
- **Linear projection(Patch embedding)**: 펼친 patch 벡터를 Transformer가 처리할 `D`차원 token으로 변환
- **Patch token**: patch 하나를 `D`차원 특징 벡터로 표현한 입력 단위
- **`[CLS]` token**: 모든 patch의 정보를 모아 최종 이미지 분류에 사용하는 학습 가능한 token
- **Positional embedding**: 각 token이 원본 이미지의 어느 위치에서 왔는지 알려 주는 위치 정보

ViT-B/16은 196개 patch token과 `[CLS]` token 하나를 합쳐 총 197개 token을 사용한다.

### 2. Transformer Encoder 내부

- **Layer Normalization(LayerNorm)**: 각 token의 특징값을 정규화하여 학습을 안정화하는 기법
- **Self-Attention**: 각 token이 다른 token을 얼마나 참고할지 Query, Key, Value로 계산하는 연산
- **Multi-Head Attention(MHA)**: 여러 attention head가 서로 다른 관점에서 token 간 관계를 병렬로 계산하고, 각 head의 결과를 결합해 다양한 특징을 학습하는 방식
- **Residual connection**: sub-layer의 입력을 출력에 더해 기존 정보와 gradient 흐름을 보존하는 연결
- **MLP(Multi-Layer Perceptron)**: attention 이후 각 token의 특징 차원을 독립적으로 변환하는 신경망

### 3. 분류 결과를 만드는 부분

- **Classification head**: 마지막 `[CLS]` 표현을 클래스별 점수로 변환하는 계층
- **Logits**: softmax를 적용하기 전의 클래스별 점수


## CNN 표현에서 ViT token 표현으로 넘어가는 경계

CNN feature map과 RGB 이미지는 모두 `(B, C, H, W)` 형태로 나타낼 수 있다.

다만 원본 ViT는 CNN feature map이 아니라 RGB 이미지를 직접 입력받으며, CNN feature map을 입력으로 사용하는 것은 hybrid 구조이다. 여기서는 두 모델의 표현 경계를 비교하기 위해 같은 NCHW 표기에서 출발한다.

높이와 너비가 `H`, `W`이고 채널 수가 `C`인 입력을 `P×P` patch로 나누면 patch 하나의 원시 픽셀 수는 `P²C`이다. 겹치지 않는 patch의 개수는 다음과 같다.

이 일반식은 `H`와 `W`가 `P`의 배수여서 가장자리에 남는 픽셀이 없다고 가정한다.

$$N = \frac{H}{P} \times \frac{W}{P}$$

각 `P²C` 벡터를 학습 가능한 선형 projection으로 길이 `D`에 맞추면 `(B, N, D)` patch token이 된다. `kernel_size=P`, `stride=P`인 `Conv2d(C, D, ...)`는 모든 patch에 같은 선형 projection을 적용하면서 `(B, D, H/P, W/P)` 격자를 한 번에 만드는 구현이다. 이 격자의 공간축을 펴고 순서를 바꾸면 `(B, N, D)`가 된다.

다음 코드는 shape 변환만 확인한다. 입력과 projection 가중치는 난수이며 학습을 수행하지 않으므로, 출력값을 학습된 특징이나 의미 있는 activation으로 해석하지 않는다.


In [ ]:
# NCHW 입력을 patch projection을 거쳐서
# ViT Token sequence로 변환되는 것을 확인

import os

# Transformers가 설치된 환경에서도 Tensorflow backend를
# 호출하지 않도록 방지하는 코드
os.environ['USE_TF'] = '0'

import torch
from torch import nn

torch.manual_seed(7)

# CNN 이미지 형태
batch, channels, height, width = 2, 3, 224, 224

patch_size = 16 # 나눠지는 이미지 한 변의 크기
embed_dim = 64 # 구조 확인용 token 특징 차원

# 가상의 이미지 생성
images = torch.randn(batch, channels, height, width)

patch_projection = nn.Conv2d(
    in_channels=channels,
    out_channels=embed_dim,
    kernel_size=patch_size,
    stride=patch_size,
)

# 입력 (B, C, 224, 224)의 각 공간축이 224/16=14로 줄어
# (B, D, 14, 14)가 된다.
patch_grid = patch_projection(images)

# 14 * 14 patch 격자를 평탄화하여 N = 196 == 전체 patch 개수
# -> (B, D, 14, 14) -> (B, D, N) == (2, 64, 196)
#  -> (B, D, N).T -> (B, N, D) == (2, 196, 64)
patch_tokens = patch_grid.flatten(start_dim=2).transpose(1, 2)

# 모든 이미지가 공유하는 하나의 전역 요약 token을 0으로 초기화
cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

# [CLS] 1개 + patch N개에 대응되도록 (1, N+1, D) 생성
position_embedding = nn.Parameter(
    torch.zeros(1, patch_tokens.shape[1] + 1, embed_dim)
)

# expand: 값을 복사하지 않고 batch 축만 B=2로 확장하며 -1은 기존 크기를 유지
cls_tokens = cls_token.expand(batch, -1, -1)

# dim=1의 token 축 앞에 `[CLS]`를 붙이고 위치 embedding을 원소별로 더해 encoder 입력 (B, N+1, D)를 만든다.
# 현재 position_embedding은 모두 0이고 학습하지 않으므로 실제 위치 차이가 아니라 ViT의 shape 계약만 재현한다.
vit_input = torch.cat([cls_tokens, patch_tokens], dim=1) + position_embedding

# 각 중간 shape를 출력해 입력 → patch grid → token sequence → encoder 입력의 변환을 대조
print("CNN/ViT 공통 NCHW 입력:", tuple(images.shape))
print("patch projection grid:", tuple(patch_grid.shape))
print("patch tokens:", tuple(patch_tokens.shape))
print("ViT encoder 입력:", tuple(vit_input.shape))

## positional embedding과 self-attention

positional embedding은 token이 원래 이미지의 어느 위치에서 왔는지를 나타내는 학습 가능한 벡터이다. self-attention은 입력 순서를 자체적으로 알지 못하므로, 동일한 patch token이라도 위치를 구분하려면 이 정보가 필요하다. ViT에서는 `[CLS]`를 포함한 `(B, N+1, D)` sequence에 같은 shape의 positional embedding을 더한 뒤 첫 encoder block으로 보낸다.

self-attention은 각 token이 다른 모든 token을 얼마나 참고할지 계산하고, 그 가중합으로 token 표현을 갱신하는 연산이다. ViT encoder 안에서 patch 사이의 장거리 관계를 반영하기 위해 사용한다. 여러 head는 서로 다른 관계를 병렬로 학습하며, attention weight의 shape는 `(B, heads, T, T)`이다. 여기서 `T=N+1`이다.

다음 CPU 코드는 앞 셀의 `(2, 197, 64)` 입력을 한 번의 multi-head self-attention에 통과시킨다. 모듈도 무작위 초기화 상태이므로 attention 값의 크기나 패턴을 의미 있게 해석하지 않고 shape와 데이터 흐름만 확인한다.


In [ ]:
self_attention = nn.MultiheadAttention(
    embed_dim=embed_dim,
    num_heads=4,
    batch_first=True,
)

attention_output, attention_weights = self_attention(
    vit_input,
    vit_input,
    vit_input,
    need_weights=True,
    average_attn_weights=False,
)

# 반환값은 갱신된 token (B, T, D)와 head를 유지한 관계 행렬 (B, heads, T, T)이다.
# 마지막 두 T축은 각각 정보를 받는 query token과 참고되는 key token의 위치를 뜻한다.
print("self-attention 출력:", tuple(attention_output.shape))
print("head별 attention weight:", tuple(attention_weights.shape))

## ViT-B/16의 `[CLS]` 표현과 분류 출력

`google/vit-base-patch16-224`의 `B/16`은 base 크기의 ViT가 16×16 patch를 사용한다는 뜻이다. 224×224 RGB 이미지 한 장은 image processor를 거쳐 `(1, 3, 224, 224)`가 되고, `3×16×16=768`개의 patch 픽셀값은 hidden size `D=768`로 투영된다. patch 196개와 `[CLS]` 하나를 합친 encoder sequence는 `(1, 197, 768)`이다. 여기서 patch의 원시 길이와 hidden size가 모두 768인 것은 이 checkpoint 설정에서의 일치이며 일반적인 규칙으로 적용하지 않는다.

`[CLS]` token은 batch마다 새로 만드는 요약 통계가 아니라 모델이 학습하는 하나의 공유 parameter이다. encoder를 통과한 최종 첫 번째 token 표현 `(B, D)`를 classification head에 넣어 `(B, 1000)` logits를 만든다. `ViTForImageClassification`의 이 CLS 표현을 임의의 mean pooling이나 별도 `pooler_output`과 혼동하지 않는다.

아래 실제 모델 경로는 `이미지 → image processor → pixel_values → ViT → logits → label` 순서이다. 공개 모델을 로컬에서 추론하므로 유료 API나 API key는 필요하지 않지만, 최초 실행에는 원격 이미지와 약 346MB 모델 파일 다운로드가 필요하다. 실행 후에는 원격 이미지와 로컬 cache의 모델에서 나온 shape와 label을 확인한다.


### 이미지 준비

실제 모델은 PIL RGB 이미지를 입력으로 받는다. 다음 셀은 [Hugging Face `google/vit-base-patch16-224` 모델 페이지](https://huggingface.co/google/vit-base-patch16-224)가 분류 예제로 사용하는 [앵무새 이미지](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png)를 내려받고 HTTP 오류를 확인한 뒤 RGB로 변환한다.

이 사진은 두 마리의 금강앵무가 화면 중심을 크게 차지하므로 ImageNet-1k의 `macaw` class와 사람이 보는 내용이 명확히 대응한다. 별도 재배포 라이선스는 확인되지 않았으므로 원격 URL로만 참조하며, 수업 외 재배포 전에는 권리를 확인한다.


In [ ]:

from io import BytesIO

import requests
from PIL import Image

IMAGE_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png"

IMAGE_URL = 'https://images.pet-friends.co.kr/v2/community/2024/11/20/7f04da3a-fef0-4bf0-8f33-cda04b3b320c.jpeg?f=webp'

response = requests.get(IMAGE_URL, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")
print("원본 이미지:", image.mode, image.size)
display(image)

### image processor와 사전학습 모델 로드

`AutoImageProcessor`는 checkpoint의 `preprocessor_config.json`을 읽어 resize, rescale, normalize 규칙을 맞춘다. `AutoModelForImageClassification`은 같은 ID에서 ViT encoder와 ImageNet-1k classification head를 불러온다. 두 객체가 같은 모델 ID를 사용해야 전처리와 가중치의 입력 계약이 일치한다.

CPU에서는 float16을 강제하지 않고 checkpoint의 기본 float32를 사용한다. `from_pretrained()`는 기본적으로 evaluation mode를 사용하지만, 아래 코드에서는 추론 의도를 분명히 하기 위해 `model.eval()`을 명시한다. 고수준 `pipeline`은 내부 단계를 가리므로 여기서는 사용하지 않는다. 최신 pipeline을 별도로 사용할 때는 `dtype` 인자를 확인하고 CPU에 `float16`을 지정하지 않는다.


In [ ]:
assert os.environ.get("USE_TF") == "0", "먼저 CPU patch projection 셀을 실행한다."

from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "google/vit-base-patch16-224"
RETURN_TENSORS = "pt"
OUTPUT_HIDDEN_STATES = True
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)
model.eval()
print(
    "model config:",
    {
        "image_size": model.config.image_size, # 24
        "patch_size": model.config.patch_size, # 16
        "hidden_size": model.config.hidden_size, # 768
        "num_labels": model.config.num_labels, # 1000
    },
)

### 전처리, ViT forward와 label 해석

image processor의 입력은 RGB PIL 이미지이고 출력은 모델이 받을 `pixel_values` tensor이다. `return_tensors="pt"`는 PyTorch tensor를 요청하며, 이 checkpoint에서는 한 장이 `(1, 3, 224, 224)`로 변환된다.

모델은 encoder의 최종 `[CLS]` 표현을 classification head에 전달해 1,000개 ImageNet class의 logit을 반환한다. logit은 확률이 아니라 정규화 전 점수이며, 가장 큰 인덱스를 `model.config.id2label`로 변환해야 사람이 읽는 label이 된다. 앞 셀의 `RETURN_TENSORS`와 `OUTPUT_HIDDEN_STATES` 설정을 사용하고, 다음 셀은 학습이 아니므로 `torch.inference_mode()`로 gradient 기록을 끈다.


In [ ]:
inputs = image_processor(images=image, return_tensors=RETURN_TENSORS)

# inference_mode() : autograd 끄기 + version 추적 끄기 == 메모리 사용량 감소
with torch.inference_mode():
    outputs = model(**inputs, output_hidden_states=OUTPUT_HIDDEN_STATES)

logits = outputs.logits
last_hidden_state = outputs.hidden_states[-1]
cls_embedding = last_hidden_state[:, 0, :]
predicted_index = logits.argmax(dim=-1).item()
predicted_label = model.config.id2label[predicted_index]

print("pixel_values:", tuple(inputs["pixel_values"].shape))
print("last hidden state:", tuple(last_hidden_state.shape))
print("CLS embedding:", tuple(cls_embedding.shape))
print("logits:", tuple(logits.shape))
print("predicted label:", predicted_label)

## 정리

CNN의 feature map과 ViT 입력은 `(B, C, H, W)` 형태를 공유할 수 있지만, 원본 ViT는 RGB 이미지를 patch로 직접 투영한다. patch projection은 공간 격자를 `(B, N, D)` token sequence로 바꾸고, `[CLS]`와 positional embedding을 결합하면 encoder 입력은 `(B, N+1, D)`가 된다.

self-attention은 각 encoder 층에서 token 사이 관계를 계산하지만 token 개수는 유지한다. 최종 `[CLS]` 표현은 이미지 전체를 분류하기 위한 head의 입력이 된다. 빠른 CPU 셀은 이 구조의 shape만 증명하며, 학습된 의미는 사전학습 checkpoint를 실제 이미지에 적용한 마지막 경로에서만 해석한다.

실행 순서는 CPU patch projection, CPU self-attention, 원격 이미지, 사전학습 모델 로드, 실제 추론 순서이다. 네트워크가 없는 환경에서는 앞의 두 CPU 셀만으로 개념과 shape를 복습할 수 있다.


In [ ]:
import torch.nn.functional as F

with torch.inference_mode():

    # (1,3,224,224) 픽셀 입력을 CLIP의 Image Encoder를 통과시켜
    # (1, C) 벡터를 만든다
    # - D = Embedding 차원 수 == 이미지를 벡터 데이터로 변환
    image_embedding = model.get_image_features(
        image_embedding = model.get_image_features(
            pixel_values=inputs["pixel_values"],
        )
    )
    # 후보 문장 (4개)를 CLIP의 Text Encoder를 통과시켜,
    # (4, D)벡터로 만든다.
    text_embedding = model.get_text_features(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

# 정규화 (p=2 == L2 Norm == 유클리드 노름)
image_embeddings = F.normalize(image_embedding, dim=-1, p=2)
text_embeddings = F.normalize(text_embedding, dim=-1, p=2)

# cos 유사도 점수 측정
cosine_scores = (image_embedding@ text_embedding.T)[0]
cosine_ranking = torch.argsort(cosine_scores, descending=True).tolist()

for rank, index in enumerate(cosine_ranking, start=1):
    print(f"{rank}. cosine={cosine_scores[index].item():.4f} | {candidate_texts[index]}")
print(f"same_top1={ranking[0] == cosine_ranking[0]}")
print(f"image_embedding_shape={tuple(image_embedding.shape)}")
print(f"text_embeddings_shape={tuple(text_embeddings.shape)}")